In [ ]:
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")
# Permite importar `creditsight` sin instalarlo: el código vive en ejemplo/src
sys.path.insert(0, str(Path.cwd().resolve().parent / "src"))


# 06 · Optimización con seguimiento (tracking)

Buscar mejores hiperparámetros es probar muchas configuraciones. Sin registro, a la mitad ya no
sabes cuál era la buena. Aquí aparece el **tracking**: cada ensayo queda guardado en MLflow con sus
parámetros, métricas y etiquetas, para poder **comparar, reproducir y auditar**.

Tres cuidados que separan una optimización seria de una ingenua:

1. **Se optimizan solo los finalistas** (random forest y LightGBM), no todo el catálogo.
2. **El espacio de búsqueda es conocimiento, no azar**: los rangos están acotados por lo que sabemos
   de cada algoritmo.
3. **La mejor puntuación de una búsqueda es optimista**: escoger el máximo de muchas pruebas sesga la
   estimación hacia arriba. Por eso reservamos clientes que la búsqueda **nunca ve** y comparamos ahí
   contra los parámetros por defecto, con un intervalo de confianza.

In [ ]:
import json
import warnings

import joblib
import matplotlib.pyplot as plt
import mlflow
import numpy as np
import optuna
import pandas as pd

from creditsight.config import GROUP_COL, PROCESSED_DIR, REPORTS_DIR, TARGET
from creditsight.data.split import group_cv, split_by_customer
from creditsight.models.train import build_pipeline, candidates, encode_target, metrics_from_proba
from creditsight.models.tuning import MODELS, bootstrap_cost_difference, tune
from creditsight.tracking.mlflow_utils import configure_mlflow, set_experiment

with warnings.catch_warnings(record=True) as caught:   # mostramos por qué se usó (o no) Databricks
    warnings.simplefilter("always")
    where = configure_mlflow()
for w in caught:
    print("Aviso:", w.message)
experiment = set_experiment("creditsight-optimizacion")
print("MLflow:", where)
print("Experimento:", experiment)

> **Si ves `local`:** no hay conexión con Databricks (sin `.env`, sin red o cuota agotada). El
> notebook funciona igual y guarda todo en `ejemplo/mlflow_local.db`. Es el respaldo previsto.

## Reservar clientes que la búsqueda no verá

Del conjunto de entrenamiento separamos el 25 % de los **clientes** (por cliente, siempre). La
búsqueda trabaja solo con el 75 %. La prueba final sigue cerrada.

In [ ]:
train = joblib.load(PROCESSED_DIR / "train.joblib")
tune_df, val_df = split_by_customer(train, test_size=0.25, seed=7)
X_tune, y_tune = tune_df.drop(columns=[TARGET]), encode_target(tune_df[TARGET])
X_val, y_val = val_df.drop(columns=[TARGET]), encode_target(val_df[TARGET])
cv = group_cv(3)
print(f"búsqueda: {tune_df.Customer_ID.nunique():,} clientes · validación reservada: {val_df.Customer_ID.nunique():,} clientes")

## La búsqueda

Optuna usa un muestreador bayesiano (TPE): en vez de probar al azar, aprende qué zonas del espacio
dieron mejor costo y explora más ahí. Cada ensayo es una **ejecución anidada** dentro de una
ejecución madre en MLflow. El presupuesto de ensayos es una decisión práctica: Databricks Free
Edition tiene cuota diaria, así que no se gasta de más.

In [ ]:
TRIALS = {"lightgbm": 20, "random forest": 12}   # presupuesto acotado a propósito
studies = {}

def run_search(name):
    studies[name] = tune(name, X_tune, y_tune, cv, n_trials=TRIALS[name])
    s = studies[name]
    print(f"{name:14s} mejor costo en la búsqueda = {s.best_value:.4f}  (ensayo {s.best_trial.number} de {TRIALS[name]})")

run_search("lightgbm")

In [ ]:
run_search("random forest")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.2), sharey=True)
for ax, (name, study) in zip(axes, studies.items()):
    values = [t.value for t in study.trials]
    ax.plot(values, "o", color="#9aa0a6", label="ensayo")
    ax.plot(np.minimum.accumulate(values), "-", color="#4c72b0", label="mejor hasta ahora")
    ax.set_title(name); ax.set_xlabel("ensayo"); ax.legend()
axes[0].set_ylabel("costo en validación cruzada (menor es mejor)")
plt.tight_layout(); plt.show()

## ¿Qué parámetros importaron?

La importancia de cada parámetro dice dónde vale la pena seguir buscando y dónde no.

In [ ]:
imp = pd.DataFrame({name: pd.Series(optuna.importance.get_param_importances(study)) for name, study in studies.items()})
imp.round(3).fillna("")

## La comprobación honesta

Comparamos, sobre los **clientes reservados**, el modelo con parámetros por defecto contra el
modelo con los mejores parámetros encontrados. Ambos se entrenan con los mismos datos de búsqueda
y se evalúan en clientes que ninguno vio. La diferencia lleva un intervalo del 95 %, remuestreando
**clientes** (no filas).

In [ ]:
defaults = {n: m for n, (m, _) in candidates().items()}
rows, probas = [], {}
for name, study in studies.items():
    _, make_fn, scale = MODELS[name]
    tuned = build_pipeline(make_fn(study.best_params), scale=scale).fit(X_tune, y_tune)
    default = build_pipeline(defaults[name], scale=scale).fit(X_tune, y_tune)
    p_tuned, p_default = tuned.predict_proba(X_val), default.predict_proba(X_val)
    probas[name] = (p_default, p_tuned)
    m_tuned, m_default = metrics_from_proba(y_val, p_tuned), metrics_from_proba(y_val, p_default)
    diff, low, high = bootstrap_cost_difference(y_val, p_tuned, p_default, val_df[GROUP_COL])
    rows.append({
        "modelo": name,
        "costo por defecto": m_default["cost"], "costo optimizado": m_tuned["cost"],
        "diferencia (opt. − defecto)": diff, "IC 95 % inferior": low, "IC 95 % superior": high,
        "log loss defecto": m_default["log_loss"], "log loss optimizado": m_tuned["log_loss"],
        "¿mejora real?": "sí" if high < 0 else ("no distinguible" if low <= 0 <= high else "empeora"),
    })
check = pd.DataFrame(rows).set_index("modelo")
check.round(4)

**Cómo leer el intervalo:** si todo el intervalo está por debajo de cero, el modelo optimizado cuesta
menos con confianza del 95 %. Si el intervalo cruza el cero, la mejora **no se distingue del ruido**
en estos clientes: no hay evidencia de que optimizar sirvió, aunque el número haya bajado.

## Lo que quedó registrado

Cada ensayo está en MLflow. Así se ven los mejores, ordenados por costo:

In [ ]:
runs = mlflow.search_runs(
    experiment_names=[experiment],
    filter_string="tags.etapa = '06_optimizacion' and metrics.cost > 0",
    order_by=["metrics.cost ASC"],
    max_results=5,
)
cols = [c for c in ["tags.modelo", "run_id", "metrics.cost", "metrics.cost_std", "metrics.log_loss", "metrics.fit_seconds"] if c in runs.columns]
runs[cols].round(4)

In [ ]:
best = {
    name: {
        "params": study.best_params,
        "cv_cost": study.best_value,
        "mlflow_parent_run_id": study.user_attrs.get("mlflow_parent_run_id"),
        "tracking": where,
    }
    for name, study in studies.items()
}
(REPORTS_DIR / "06_best_params.json").write_text(json.dumps(best, indent=2, default=float), encoding="utf-8")
check.to_csv(REPORTS_DIR / "06_comprobacion.csv")
print("parámetros guardados en reports/06_best_params.json")